# Own text rather than uploading file

In [ ]:
# ============================================================
# RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA
# ============================================================


# ============================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ============================================================

import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# Check whether OpenAI API key exists
if os.environ["OPENAI_API_KEY"]:
    print("API Key is set.")


# ============================================================
# 2. LOAD THE LLM
# ============================================================

from langchain_openai import ChatOpenAI

# Create the LLM
llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0
)


# ============================================================
# STEP 1: PREPARING DOCUMENT FOR YOUR TEXT
# ============================================================

from langchain_core.documents import Document


# Your own text data
my_text = """
Artificial intelligence (AI) is the capability of computational
systems to perform tasks typically associated with human intelligence,
such as learning, reasoning, problem-solving, perception, and
decision-making. It is a field of research in computer science that
develops and studies methods and software that enable machines to
perceive their environment and use learning and intelligence to take
actions that maximize their chances of achieving defined goals.

High-profile applications of AI include advanced web search engines
(e.g., Google Search); recommendation systems (used by YouTube,
Amazon, and Netflix); virtual assistants (e.g., Google Assistant,
Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and
creative tools (e.g., language models and AI art); and superhuman
play and analysis in strategy games (e.g., chess and Go).

Various subfields of AI research are centered around particular goals
and the use of particular tools. The traditional goals of AI research
include learning, reasoning, knowledge representation, planning,
natural language processing, perception, and support for robotics.

To reach these goals, AI researchers have adapted and integrated a
wide range of techniques, including search and mathematical
optimization, formal logic, artificial neural networks, and methods
based on statistics, operations research, and economics.

AI also draws upon psychology, linguistics, philosophy, neuroscience,
and other fields. Some companies, such as OpenAI, Google DeepMind and
Meta, aim to create artificial general intelligence (AGI) – AI that
can complete virtually any cognitive task at least as well as a human.
"""


# Convert our text into a LangChain Document
docs = [
    Document(
        page_content=my_text,
        metadata={
            "source": "ABC",
            "documentID": "Doc1"
        }
    )
]

print(docs)


# ============================================================
# STEP 2: SPLITTING THE DOCUMENT INTO CHUNKS
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create a text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)


# Split the document into smaller chunks
chunks = splitter.split_documents(docs)

print(chunks)


# ============================================================
# STEP 3: CREATING EMBEDDINGS FOR THE CHUNKS
# ============================================================

from langchain_openai import OpenAIEmbeddings


# Create embedding model
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# STEP 4: CREATE AND STORE EMBEDDINGS IN VECTOR STORE
# ============================================================

from langchain_community.vectorstores import Chroma


# Create Chroma vector store
# The chunks are converted into embeddings
# and stored in the vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)


# ============================================================
# STEP 5: SEMANTIC SEARCH
# ============================================================

# Search for the chunks most relevant to the question
results = vectorstore.similarity_search(
    "What is AI?",
    k=3
)

print(results)


# ============================================================
# TALK TO LLM
# ============================================================

# Retrieve relevant chunks from the vector store
context = vectorstore.similarity_search(
    "What is AI?",
    k=3
)


# Send the question + retrieved context to the LLM
response = llm.invoke(
    f"""
    What is AI?

    You can answer using the following context:

    {context}
    """
)


# Print the final answer
print(response.content)